# Calculate Dome resolution using MeshMap

**Local digital resolution** = how many projector pixels represent one
nominal degree at each part of the dome.

To determine this, at each mesh location, the script calculates a Jacobian: a local conversion between movement in BonVision degrees and movement in projector pixels. It uses this to calculate sampling in every direction and reports the lowest value, giving the worst-case local pixels per degree

That value is then checked against two rules to see what visual gratings can be represented:

- **2 pixels/cycle:** theoretical Nyquist boundary (i.e. one pixel for black and one pixel for white)
- **4 pixels/cycle:** 2x Nyquist (safer sampling requirement)

Sampling can differ with direction. The main maps use the lowest local
value, so they do not assume a particular grating orientation.

## 1. Imports

In [ ]:
from datetime import UTC, datetime
import hashlib
from importlib import reload
from importlib.metadata import version
import json
from pathlib import Path
import platform
import sys

from IPython.display import display
import matplotlib.pyplot as plt
from matplotlib.colors import BoundaryNorm, ListedColormap
from matplotlib.patches import Patch
import numpy as np
import pandas as pd
import plotly.graph_objects as go


search_roots = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
ANALYSIS_DIRECTORY = next(
    (
        directory
        for root in search_roots
        for directory in (root, root / "src" / "MeshMapping")
        if (directory / "analyse_meshmap_resolution.py").is_file()
    ),
    None,
)
if ANALYSIS_DIRECTORY is None:
    raise FileNotFoundError(
        "Could not find analyse_meshmap_resolution.py. "
        "Start Jupyter from the av-dome repository."
    )

if str(ANALYSIS_DIRECTORY) not in sys.path:
    sys.path.insert(0, str(ANALYSIS_DIRECTORY))

import analyse_meshmap_resolution

reload(analyse_meshmap_resolution)
from analyse_meshmap_resolution import (
    ANALYSIS_VERSION,
    build_mesh_grid,
    calculate_grating_pixels_per_degree,
    calculate_resolution,
    nominal_solid_angle_weights,
    read_mesh_map,
)

print(f"Analysis directory: {ANALYSIS_DIRECTORY}")

## 2. Settings

Later figures repeat the key params for that cell at the top of the cell,
so you can change and rerun that figure independently.

In [ ]:
# Input
MESH_MAP_PATH = Path("path/to/final_mesh_map.xlsx")
MESH_SHEET_NAME = None  # First sheet

# Projector and BonVision view: must match src/main.bonsai
PROJECTOR_WIDTH_PIXELS = 1920
PROJECTOR_HEIGHT_PIXELS = 1080
AZIMUTH_RANGE_DEGREES = (-135.0, 135.0)
ELEVATION_RANGE_DEGREES = (-22.5, 82.5)

# Grating check
SPATIAL_FREQUENCY_CPD = 0.30 # the grating frequency you want to test in cycles per degree.
NYQUIST_PIXELS_PER_CYCLE = 2.0
SAFETY_PIXELS_PER_CYCLE = 4.0

# BonVision DrawGratings angle: 0° = vertical; 90° = horizontal bars
GRATING_ANGLE_DEGREES = 90.0

# Starting view for the curved dome
DOME_CAMERA_AZIMUTH_DEGREES = 0
DOME_CAMERA_ELEVATION_DEGREES = 0
DOME_CAMERA_DISTANCE = 1.8
SHOW_DOME_GRID_LINES = True
DOME_GRID_LINE_SPACING = 3

# Location to inspect
INSPECT_AZIMUTH_DEGREES = 0.0
INSPECT_ELEVATION_DEGREES = 0.0

# Coverage curve
CPD_CURVE_RANGE = (0.01, 1.50)
CURVE_POINT_COUNT = 250

# Plotting and saving
CONTINUOUS_COLOUR_MAP = "viridis"
ADEQUATE_COLOUR = "#238b57"
UNDERSAMPLED_COLOUR = "#c83e35"
SAVE_FIGURES = False
OUTPUT_DIRECTORY = ANALYSIS_DIRECTORY / "outputs"

if SPATIAL_FREQUENCY_CPD <= 0:
    raise ValueError("SPATIAL_FREQUENCY_CPD must be positive.")
if NYQUIST_PIXELS_PER_CYCLE <= 0:
    raise ValueError("THEORETICAL_PIXELS_PER_CYCLE must be positive.")
if SAFETY_PIXELS_PER_CYCLE <= NYQUIST_PIXELS_PER_CYCLE:
    raise ValueError(
        "SAFETY_PIXELS_PER_CYCLE must exceed "
        "THEORETICAL_PIXELS_PER_CYCLE."
    )
projector_dimensions = (
    PROJECTOR_WIDTH_PIXELS,
    PROJECTOR_HEIGHT_PIXELS,
)
if any(
    isinstance(value, bool) or not isinstance(value, int)
    for value in projector_dimensions
):
    raise TypeError("Projector dimensions must be integers.")
if any(value <= 0 for value in projector_dimensions):
    raise ValueError("Projector dimensions must be positive.")
if AZIMUTH_RANGE_DEGREES[1] <= AZIMUTH_RANGE_DEGREES[0]:
    raise ValueError("AZIMUTH_RANGE_DEGREES must increase.")
if ELEVATION_RANGE_DEGREES[1] <= ELEVATION_RANGE_DEGREES[0]:
    raise ValueError("ELEVATION_RANGE_DEGREES must increase.")
if (
    CPD_CURVE_RANGE[0] <= 0
    or CPD_CURVE_RANGE[1] <= CPD_CURVE_RANGE[0]
):
    raise ValueError("CPD_CURVE_RANGE must be positive and increase.")
if (
    isinstance(CURVE_POINT_COUNT, bool)
    or not isinstance(CURVE_POINT_COUNT, int)
):
    raise TypeError("CURVE_POINT_COUNT must be an integer.")
if CURVE_POINT_COUNT < 2:
    raise ValueError("CURVE_POINT_COUNT must be at least 2.")
if (
    isinstance(DOME_GRID_LINE_SPACING, bool)
    or not isinstance(DOME_GRID_LINE_SPACING, int)
):
    raise TypeError("DOME_GRID_LINE_SPACING must be an integer.")
if DOME_GRID_LINE_SPACING < 1:
    raise ValueError("DOME_GRID_LINE_SPACING must be at least 1.")
if DOME_CAMERA_DISTANCE <= 0:
    raise ValueError("DOME_CAMERA_DISTANCE must be positive.")

## 3. Calculate local resolution

This is the only underlying resolution measurement in the notebook.

The other quantities are just conversions:

`pixels per cycle = pixels per degree ÷ cycles per degree`

`boundary frequency = pixels per degree ÷ required pixels per cycle`

In [ ]:
mesh_path = MESH_MAP_PATH.expanduser().resolve()
if not mesh_path.is_file():
    raise FileNotFoundError(f"Mesh map not found: {mesh_path}")

mesh_table, selected_sheet_name = read_mesh_map(
    mesh_path,
    MESH_SHEET_NAME,
)
mesh_grid = build_mesh_grid(mesh_table)
resolution = calculate_resolution(
    mesh_grid,
    width=PROJECTOR_WIDTH_PIXELS,
    height=PROJECTOR_HEIGHT_PIXELS,
    azimuth_range=AZIMUTH_RANGE_DEGREES,
    elevation_range=ELEVATION_RANGE_DEGREES,
    evaluated_cpd=SPATIAL_FREQUENCY_CPD,
)

cell_pixels_per_degree = np.min(
    resolution.candidate_minimum_nominal_pixels_per_degree,
    axis=0,
)

def calculate_sampling_checks(
    pixels_per_degree,
    spatial_frequency_cpd,
    theoretical_pixels_per_cycle,
    safety_pixels_per_cycle,
):
    if spatial_frequency_cpd <= 0:
        raise ValueError("Spatial frequency must be positive.")
    if theoretical_pixels_per_cycle <= 0:
        raise ValueError("Pixels per cycle must be positive.")
    if safety_pixels_per_cycle <= theoretical_pixels_per_cycle:
        raise ValueError("2x Nyquist must exceed Nyquist.")

    pixels_per_cycle = pixels_per_degree / spatial_frequency_cpd
    satisfies_nyquist = (
        pixels_per_cycle > theoretical_pixels_per_cycle
    )
    meets_safety_criterion = (
        pixels_per_cycle >= safety_pixels_per_cycle
    )
    return (
        pixels_per_cycle,
        satisfies_nyquist,
        meets_safety_criterion,
    )

cell_solid_angle_weights = nominal_solid_angle_weights(resolution)

cell_azimuth_centres = (
    resolution.azimuth_vertices[:-1]
    + resolution.azimuth_vertices[1:]
) / 2
cell_elevation_centres = (
    resolution.nominal_elevation_vertices[:-1]
    + resolution.nominal_elevation_vertices[1:]
) / 2

In [ ]:
# Parameters for this summary
SPATIAL_FREQUENCY_CPD = 0.30
NYQUIST_PIXELS_PER_CYCLE = 2.0
SAFETY_PIXELS_PER_CYCLE = 4.0

summary_parameters = {
    "spatial_frequency_cpd": SPATIAL_FREQUENCY_CPD,
    "theoretical_pixels_per_cycle": NYQUIST_PIXELS_PER_CYCLE,
    "safety_pixels_per_cycle": SAFETY_PIXELS_PER_CYCLE,
}

(
    cell_pixels_per_cycle,
    cell_satisfies_nyquist_sampling,
    cell_meets_safety_criterion,
) = calculate_sampling_checks(
    cell_pixels_per_degree,
    SPATIAL_FREQUENCY_CPD,
    NYQUIST_PIXELS_PER_CYCLE,
    SAFETY_PIXELS_PER_CYCLE,
)

resolution_summary = pd.Series(
    {
        "minimum pixels/nominal degree": cell_pixels_per_degree.min(),
        "median pixels/nominal degree": np.median(cell_pixels_per_degree),
        "maximum pixels/nominal degree": cell_pixels_per_degree.max(),
    },
    name="value",
)

sampling_summary = pd.DataFrame(
    {
        "rule": [
            "Nyquist boundary",
            "2x Nyquist",
        ],
        "required pixels/cycle": [
            NYQUIST_PIXELS_PER_CYCLE,
            SAFETY_PIXELS_PER_CYCLE,
        ],
        "required pixels/degree": [
            SPATIAL_FREQUENCY_CPD
            * NYQUIST_PIXELS_PER_CYCLE,
            SPATIAL_FREQUENCY_CPD
            * SAFETY_PIXELS_PER_CYCLE,
        ],
        "dome surface satisfying check (%)": [
            100 * np.average(
                cell_satisfies_nyquist_sampling,
                weights=cell_solid_angle_weights,
            ),
            100 * np.average(
                cell_meets_safety_criterion,
                weights=cell_solid_angle_weights,
            ),
        ],
    }
)

print(f"Grating tested: {SPATIAL_FREQUENCY_CPD:g} cpd")
display(resolution_summary.to_frame().round(1))
display(sampling_summary.round(1))

## 4. Projector footprint

This shows which part of the full projector image is mapped onto the dome.

In [ ]:
projector_footprint_figure, projector_footprint_axis = plt.subplots(
    figsize=(10, 5.5),
    constrained_layout=True,
)
projector_footprint_axis.plot(
    resolution.projector_pixel_x.T,
    resolution.projector_pixel_y.T,
    color="#2a6f97",
    linewidth=0.6,
)
projector_footprint_axis.plot(
    resolution.projector_pixel_x,
    resolution.projector_pixel_y,
    color="#2a6f97",
    linewidth=0.6,
)
projector_footprint_axis.set(
    title=(
        "Dome footprint in the "
        f"{PROJECTOR_WIDTH_PIXELS} × {PROJECTOR_HEIGHT_PIXELS} "
        "projector output"
    ),
    xlabel="Projector X (pixels)",
    ylabel="Projector Y (pixels)",
    xlim=(0, PROJECTOR_WIDTH_PIXELS),
    ylim=(PROJECTOR_HEIGHT_PIXELS, 0),
    aspect="equal",
)
plt.xticks(np.arange(0, PROJECTOR_WIDTH_PIXELS + 1, 1920/2))
plt.yticks(np.arange(0, PROJECTOR_HEIGHT_PIXELS + 1, 1080/2))
plt.show()

# Based on this image:

# horizontal grating should be represented worst at the center bc each cell is most flattened here
# horizontal grating should be represented best at the sides because the cells are least flattened here

# vertical grating should be best at center because its most stretched sideways here
# vertical grating should be worst at sides because its most flattened sideways here


## Plotting helper

The interactive dome appears first in each section. Drag to rotate and scroll to zoom.

In [ ]:
def calculate_dome_vertices(resolution_grid):
    azimuth_grid, elevation_grid = np.meshgrid(
        np.deg2rad(resolution_grid.azimuth_vertices),
        np.deg2rad(resolution_grid.nominal_elevation_vertices),
    )
    left_right = np.cos(elevation_grid) * np.sin(azimuth_grid)
    front_back = np.cos(elevation_grid) * np.cos(azimuth_grid)
    up_down = np.sin(elevation_grid)
    return left_right, front_back, up_down


def calculate_camera_eye(
    azimuth_degrees,
    elevation_degrees,
    distance,
):
    azimuth_radians = np.deg2rad(azimuth_degrees)
    elevation_radians = np.deg2rad(elevation_degrees)
    return {
        "x": (
            distance
            * np.cos(elevation_radians)
            * np.cos(azimuth_radians)
        ),
        "y": (
            distance
            * np.cos(elevation_radians)
            * np.sin(azimuth_radians)
        ),
        "z": distance * np.sin(elevation_radians),
    }


def calculate_dome_triangles(row_count, column_count):
    vertex_indices = np.arange(
        row_count * column_count
    ).reshape(row_count, column_count)

    top_left = vertex_indices[:-1, :-1].ravel()
    top_right = vertex_indices[:-1, 1:].ravel()
    bottom_right = vertex_indices[1:, 1:].ravel()
    bottom_left = vertex_indices[1:, :-1].ravel()

    triangle_i = np.concatenate([top_left, top_left])
    triangle_j = np.concatenate([top_right, bottom_right])
    triangle_k = np.concatenate([bottom_right, bottom_left])
    return triangle_i, triangle_j, triangle_k


def add_dome_grid_lines(
    figure,
    left_right,
    front_back,
    up_down,
    line_spacing,
):
    line_style = {
        "color": "rgba(40, 40, 40, 0.25)",
        "width": 1,
    }
    for row_index in range(
        0,
        left_right.shape[0],
        line_spacing,
    ):
        figure.add_trace(
            go.Scatter3d(
                x=left_right[row_index, :],
                y=front_back[row_index, :],
                z=up_down[row_index, :],
                mode="lines",
                line=line_style,
                hoverinfo="skip",
                showlegend=False,
            )
        )
    for column_index in range(
        0,
        left_right.shape[1],
        line_spacing,
    ):
        figure.add_trace(
            go.Scatter3d(
                x=left_right[:, column_index],
                y=front_back[:, column_index],
                z=up_down[:, column_index],
                mode="lines",
                line=line_style,
                hoverinfo="skip",
                showlegend=False,
            )
        )


def build_interactive_dome(
    resolution_grid,
    cell_values,
    title,
    colour_scale,
    colour_minimum,
    colour_maximum,
    colourbar_title,
    hover_label,
    colourbar_tick_values=None,
    colourbar_tick_text=None,
):
    left_right, front_back, up_down = calculate_dome_vertices(
        resolution_grid
    )
    triangle_i, triangle_j, triangle_k = calculate_dome_triangles(
        *left_right.shape
    )
    triangle_values = np.concatenate(
        [cell_values.ravel(), cell_values.ravel()]
    )

    colourbar_settings = {"title": colourbar_title}
    if colourbar_tick_values is not None:
        colourbar_settings["tickvals"] = colourbar_tick_values
        colourbar_settings["ticktext"] = colourbar_tick_text

    figure = go.Figure(
        go.Mesh3d(
            x=left_right.ravel(),
            y=front_back.ravel(),
            z=up_down.ravel(),
            i=triangle_i,
            j=triangle_j,
            k=triangle_k,
            intensity=triangle_values,
            intensitymode="cell",
            colorscale=colour_scale,
            cmin=colour_minimum,
            cmax=colour_maximum,
            colorbar=colourbar_settings,
            flatshading=True,
            lighting={
                "ambient": 0.9,
                "diffuse": 0.25,
                "specular": 0.0,
            },
            hovertemplate=(
                f"{hover_label}: %{{intensity:.3f}}"
                "<extra></extra>"
            ),
            showscale=True,
        )
    )

    if SHOW_DOME_GRID_LINES:
        add_dome_grid_lines(
            figure,
            left_right,
            front_back,
            up_down,
            DOME_GRID_LINE_SPACING,
        )

    figure.add_trace(
        go.Scatter3d(
            x=[0],
            y=[0],
            z=[0],
            mode="markers+text",
            marker={"size": 4, "color": "#222222"},
            text=["mouse"],
            textposition="bottom center",
            hoverinfo="skip",
            showlegend=False,
        )
    )
    figure.update_layout(
        title=f"{title}<br><sup>Drag to rotate; scroll to zoom</sup>",
        height=650,
        margin={"l": 0, "r": 0, "t": 75, "b": 0},
        scene={
            "aspectmode": "data",
            "dragmode": "orbit",
            "camera": {
                "eye": calculate_camera_eye(
                    DOME_CAMERA_AZIMUTH_DEGREES,
                    DOME_CAMERA_ELEVATION_DEGREES,
                    DOME_CAMERA_DISTANCE,
                )
            },
            "xaxis": {"visible": False},
            "yaxis": {"visible": False},
            "zaxis": {"visible": False},
        },
    )
    return figure

## 5. Local digital resolution

Each colour answers:

**At this part of visual space, how many projector pixels represent one
nominal degree in the least well sampled direction?**

Degrees refer to BonVision coordinates, not measured visual angle at the mouse's eye.
They should approximate visual-angle degrees if the rig geometry and calibration are accurate.

The rectangle is an unwrapped view of the curved dome.

The first pair of plots shows the lowest sampling across all possible
grating orientations. The following pairs show horizontal bars
(BonVision angle 90°) and vertical bars (BonVision angle 0°).

In [ ]:
# Parameters for this figure
DOME_CAMERA_AZIMUTH_DEGREES = 0
DOME_CAMERA_ELEVATION_DEGREES = 0
DOME_CAMERA_DISTANCE = 1.8
SHOW_DOME_GRID_LINES = True
DOME_GRID_LINE_SPACING = 3
CONTINUOUS_COLOUR_MAP = "viridis"
HORIZONTAL_GRATING_ANGLE_DEGREES = 90.0
VERTICAL_GRATING_ANGLE_DEGREES = 0.0

local_resolution_parameters = {
    "camera_azimuth_degrees": DOME_CAMERA_AZIMUTH_DEGREES,
    "camera_elevation_degrees": DOME_CAMERA_ELEVATION_DEGREES,
    "camera_distance": DOME_CAMERA_DISTANCE,
    "show_grid_lines": SHOW_DOME_GRID_LINES,
    "grid_line_spacing": DOME_GRID_LINE_SPACING,
    "colour_map": CONTINUOUS_COLOUR_MAP,
    "horizontal_grating_angle_degrees": (
        HORIZONTAL_GRATING_ANGLE_DEGREES
    ),
    "vertical_grating_angle_degrees": (
        VERTICAL_GRATING_ANGLE_DEGREES
    ),
}

local_resolution_dome = build_interactive_dome(
    resolution,
    cell_pixels_per_degree,
    title="Local digital resolution: worst orientation",
    colour_scale=CONTINUOUS_COLOUR_MAP,
    colour_minimum=float(cell_pixels_per_degree.min()),
    colour_maximum=float(cell_pixels_per_degree.max()),
    colourbar_title="pixels/degree",
    hover_label="pixels/degree",
)
local_resolution_dome.show()

local_resolution_figure, local_resolution_axis = plt.subplots(
    figsize=(10, 5.5),
    constrained_layout=True,
)
local_resolution_image = local_resolution_axis.pcolormesh(
    resolution.azimuth_vertices,
    resolution.nominal_elevation_vertices,
    cell_pixels_per_degree,
    shading="flat",
    cmap=CONTINUOUS_COLOUR_MAP,
)
local_resolution_axis.set(
    title="Local digital resolution: worst orientation",
    xlabel="Nominal azimuth (degrees)",
    ylabel="Nominal elevation (degrees)",
)
local_resolution_figure.colorbar(
    local_resolution_image,
    ax=local_resolution_axis,
    label="pixels per nominal degree",
)
plt.show()

horizontal_grating_pixels_per_degree = (
    calculate_grating_pixels_per_degree(
        resolution,
        grating_angle_degrees=HORIZONTAL_GRATING_ANGLE_DEGREES,
    )
)

horizontal_grating_resolution_dome = build_interactive_dome(
    resolution,
    horizontal_grating_pixels_per_degree,
    title="Horizontal grating",
    colour_scale=CONTINUOUS_COLOUR_MAP,
    colour_minimum=float(
        horizontal_grating_pixels_per_degree.min()
    ),
    colour_maximum=float(
        horizontal_grating_pixels_per_degree.max()
    ),
    colourbar_title="pixels/degree",
    hover_label="pixels/degree",
)
horizontal_grating_resolution_dome.show()

horizontal_grating_resolution_figure, horizontal_grating_resolution_axis = (
    plt.subplots(
        figsize=(10, 5.5),
        constrained_layout=True,
    )
)
horizontal_grating_resolution_image = (
    horizontal_grating_resolution_axis.pcolormesh(
        resolution.azimuth_vertices,
        resolution.nominal_elevation_vertices,
        horizontal_grating_pixels_per_degree,
        shading="flat",
        cmap=CONTINUOUS_COLOUR_MAP,
    )
)
horizontal_grating_resolution_axis.set(
    title="Horizontal grating",
    xlabel="Nominal azimuth (degrees)",
    ylabel="Nominal elevation (degrees)",
)
horizontal_grating_resolution_figure.colorbar(
    horizontal_grating_resolution_image,
    ax=horizontal_grating_resolution_axis,
    label="pixels per nominal degree",
)
plt.show()

vertical_grating_pixels_per_degree = (
    calculate_grating_pixels_per_degree(
        resolution,
        grating_angle_degrees=VERTICAL_GRATING_ANGLE_DEGREES,
    )
)

vertical_grating_resolution_dome = build_interactive_dome(
    resolution,
    vertical_grating_pixels_per_degree,
    title="Vertical grating",
    colour_scale=CONTINUOUS_COLOUR_MAP,
    colour_minimum=float(
        vertical_grating_pixels_per_degree.min()
    ),
    colour_maximum=float(
        vertical_grating_pixels_per_degree.max()
    ),
    colourbar_title="pixels/degree",
    hover_label="pixels/degree",
)
vertical_grating_resolution_dome.show()

vertical_grating_resolution_figure, vertical_grating_resolution_axis = (
    plt.subplots(
        figsize=(10, 5.5),
        constrained_layout=True,
    )
)
vertical_grating_resolution_image = (
    vertical_grating_resolution_axis.pcolormesh(
        resolution.azimuth_vertices,
        resolution.nominal_elevation_vertices,
        vertical_grating_pixels_per_degree,
        shading="flat",
        cmap=CONTINUOUS_COLOUR_MAP,
    )
)
vertical_grating_resolution_axis.set(
    title="Vertical grating",
    xlabel="Nominal azimuth (degrees)",
    ylabel="Nominal elevation (degrees)",
)
vertical_grating_resolution_figure.colorbar(
    vertical_grating_resolution_image,
    ax=vertical_grating_resolution_axis,
    label="pixels per nominal degree",
)
plt.show()

## 6. Does the selected grating satisfy each check?

These maps apply the two sampling rules to the same local-resolution
values. Green means that the selected spatial frequency satisfies the
check; red means that it does not.

The percentage is weighted by solid angle and represents the fraction
of the dome surface.

In [ ]:
# Parameters for this figure
SPATIAL_FREQUENCY_CPD = 0.3
NYQUIST_PIXELS_PER_CYCLE = 2.0
SAFETY_PIXELS_PER_CYCLE = 4.0
DOME_CAMERA_AZIMUTH_DEGREES = 0
DOME_CAMERA_ELEVATION_DEGREES = 0
DOME_CAMERA_DISTANCE = 1.8
SHOW_DOME_GRID_LINES = True
DOME_GRID_LINE_SPACING = 3
ADEQUATE_COLOUR = "#238b57"
UNDERSAMPLED_COLOUR = "#c83e35"

sampling_check_parameters = {
    "spatial_frequency_cpd": SPATIAL_FREQUENCY_CPD,
    "theoretical_pixels_per_cycle": NYQUIST_PIXELS_PER_CYCLE,
    "safety_pixels_per_cycle": SAFETY_PIXELS_PER_CYCLE,
    "camera_azimuth_degrees": DOME_CAMERA_AZIMUTH_DEGREES,
    "camera_elevation_degrees": DOME_CAMERA_ELEVATION_DEGREES,
    "camera_distance": DOME_CAMERA_DISTANCE,
    "show_grid_lines": SHOW_DOME_GRID_LINES,
    "grid_line_spacing": DOME_GRID_LINE_SPACING,
    "adequate_colour": ADEQUATE_COLOUR,
    "undersampled_colour": UNDERSAMPLED_COLOUR,
}

(
    cell_pixels_per_cycle,
    cell_satisfies_nyquist_sampling,
    cell_meets_safety_criterion,
) = calculate_sampling_checks(
    cell_pixels_per_degree,
    SPATIAL_FREQUENCY_CPD,
    NYQUIST_PIXELS_PER_CYCLE,
    SAFETY_PIXELS_PER_CYCLE,
)

theoretical_dome_surface_percentage = 100 * np.average(
    cell_satisfies_nyquist_sampling,
    weights=cell_solid_angle_weights,
)
safety_dome_surface_percentage = 100 * np.average(
    cell_meets_safety_criterion,
    weights=cell_solid_angle_weights,
)

pass_fail_plotly_colour_scale = [
    [0.0, UNDERSAMPLED_COLOUR],
    [0.499, UNDERSAMPLED_COLOUR],
    [0.5, ADEQUATE_COLOUR],
    [1.0, ADEQUATE_COLOUR],
]

theoretical_limit_dome = build_interactive_dome(
    resolution,
    cell_satisfies_nyquist_sampling.astype(float),
    title=(
        f"{NYQUIST_PIXELS_PER_CYCLE:g} pixels/cycle<br>"
        f"{theoretical_dome_surface_percentage:.1f}% "
        "of dome surface"
    ),
    colour_scale=pass_fail_plotly_colour_scale,
    colour_minimum=0,
    colour_maximum=1,
    colourbar_title="result",
    hover_label="pass",
    colourbar_tick_values=[0, 1],
    colourbar_tick_text=["does not satisfy", "satisfies"],
)
theoretical_limit_dome.show()

safety_criterion_dome = build_interactive_dome(
    resolution,
    cell_meets_safety_criterion.astype(float),
    title=(
        f"{SAFETY_PIXELS_PER_CYCLE:g} pixels/cycle<br>"
        f"{safety_dome_surface_percentage:.1f}% "
        "of dome surface"
    ),
    colour_scale=pass_fail_plotly_colour_scale,
    colour_minimum=0,
    colour_maximum=1,
    colourbar_title="result",
    hover_label="pass",
    colourbar_tick_values=[0, 1],
    colourbar_tick_text=["does not satisfy", "satisfies"],
)
safety_criterion_dome.show()

pass_fail_colour_map = ListedColormap(
    [UNDERSAMPLED_COLOUR, ADEQUATE_COLOUR]
)
pass_fail_colour_scale = BoundaryNorm(
    [-0.5, 0.5, 1.5],
    pass_fail_colour_map.N,
)

sampling_check_figure, sampling_check_axes = plt.subplots(
    1,
    2,
    figsize=(13, 5.5),
    constrained_layout=True,
    sharex=True,
    sharey=True,
)

sampling_checks = [
    (
        cell_satisfies_nyquist_sampling,
        NYQUIST_PIXELS_PER_CYCLE,
        theoretical_dome_surface_percentage,
    ),
    (
        cell_meets_safety_criterion,
        SAFETY_PIXELS_PER_CYCLE,
        safety_dome_surface_percentage,
    ),
]

for axis, (
    passing_cells,
    required_pixels_per_cycle,
    dome_surface_percentage,
) in zip(
    sampling_check_axes,
    sampling_checks,
):
    axis.pcolormesh(
        resolution.azimuth_vertices,
        resolution.nominal_elevation_vertices,
        passing_cells.astype(int),
        shading="flat",
        cmap=pass_fail_colour_map,
        norm=pass_fail_colour_scale,
    )
    axis.set(
        title=(
            f"{required_pixels_per_cycle:g} pixels/cycle\n"
            f"{dome_surface_percentage:.1f}% of dome surface"
        ),
        xlabel="Nominal azimuth (degrees)",
        ylabel="Nominal elevation (degrees)",
    )

sampling_check_figure.suptitle(
    f"Can the dome sample a {SPATIAL_FREQUENCY_CPD:g} cpd grating?"
)
sampling_check_figure.legend(
    handles=[
        Patch(color=UNDERSAMPLED_COLOUR, label="Does not satisfy"),
        Patch(color=ADEQUATE_COLOUR, label="Satisfies"),
    ],
    loc="outside lower center",
    ncols=2,
)
plt.show()

## 7. Convert local resolution into frequency boundaries

This is just the local resolution divided by either 2 or 4 pixels/cycle.

First map shows Nyquist at 2 pixels/cycle. Second map shows 2x Nyquist at 4 pixels/cycle.

In [ ]:
# Parameters for this figure
NYQUIST_PIXELS_PER_CYCLE = 2.0
SAFETY_PIXELS_PER_CYCLE = 4.0
COLOUR_MINIMUM_CPD = 0.0
COLOUR_MAXIMUM_CPD = 0.5
DOME_CAMERA_AZIMUTH_DEGREES = 0
DOME_CAMERA_ELEVATION_DEGREES = 0
DOME_CAMERA_DISTANCE = 1.8
SHOW_DOME_GRID_LINES = True
DOME_GRID_LINE_SPACING = 3
CONTINUOUS_COLOUR_MAP = "viridis"

frequency_boundary_parameters = {
    "theoretical_pixels_per_cycle": NYQUIST_PIXELS_PER_CYCLE,
    "safety_pixels_per_cycle": SAFETY_PIXELS_PER_CYCLE,
    "colour_minimum_cpd": COLOUR_MINIMUM_CPD,
    "colour_maximum_cpd": COLOUR_MAXIMUM_CPD,
    "camera_azimuth_degrees": DOME_CAMERA_AZIMUTH_DEGREES,
    "camera_elevation_degrees": DOME_CAMERA_ELEVATION_DEGREES,
    "camera_distance": DOME_CAMERA_DISTANCE,
    "show_grid_lines": SHOW_DOME_GRID_LINES,
    "grid_line_spacing": DOME_GRID_LINE_SPACING,
    "colour_map": CONTINUOUS_COLOUR_MAP,
}

if COLOUR_MINIMUM_CPD >= COLOUR_MAXIMUM_CPD:
    raise ValueError(
        "COLOUR_MINIMUM_CPD must be below COLOUR_MAXIMUM_CPD."
    )

cell_nyquist_boundary_cpd = (
    cell_pixels_per_degree / NYQUIST_PIXELS_PER_CYCLE
)
cell_safety_criterion_cpd = (
    cell_pixels_per_degree / SAFETY_PIXELS_PER_CYCLE
)

nyquist_frequency_boundary_dome = build_interactive_dome(
    resolution,
    cell_nyquist_boundary_cpd,
    title=f"{NYQUIST_PIXELS_PER_CYCLE:g} pixels/cycle",
    colour_scale=CONTINUOUS_COLOUR_MAP,
    colour_minimum=COLOUR_MINIMUM_CPD,
    colour_maximum=COLOUR_MAXIMUM_CPD,
    colourbar_title="cycles/degree",
    hover_label="cycles/degree",
)
nyquist_frequency_boundary_dome.show()

safety_frequency_boundary_dome = build_interactive_dome(
    resolution,
    cell_safety_criterion_cpd,
    title=f"{SAFETY_PIXELS_PER_CYCLE:g} pixels/cycle",
    colour_scale=CONTINUOUS_COLOUR_MAP,
    colour_minimum=COLOUR_MINIMUM_CPD,
    colour_maximum=COLOUR_MAXIMUM_CPD,
    colourbar_title="cycles/degree",
    hover_label="cycles/degree",
)
safety_frequency_boundary_dome.show()

frequency_boundary_figure, frequency_boundary_axes = plt.subplots(
    1,
    2,
    figsize=(13, 5.5),
    constrained_layout=True,
    sharex=True,
    sharey=True,
)

frequency_boundary_maps = [
    (
        cell_nyquist_boundary_cpd,
        f"{NYQUIST_PIXELS_PER_CYCLE:g} pixels/cycle",
    ),
    (
        cell_safety_criterion_cpd,
        f"{SAFETY_PIXELS_PER_CYCLE:g} pixels/cycle",
    ),
]

for axis, (frequency_boundary_cpd, title) in zip(
    frequency_boundary_axes,
    frequency_boundary_maps,
):
    frequency_boundary_image = axis.pcolormesh(
        resolution.azimuth_vertices,
        resolution.nominal_elevation_vertices,
        frequency_boundary_cpd,
        shading="flat",
        cmap=CONTINUOUS_COLOUR_MAP,
        vmin=COLOUR_MINIMUM_CPD,
        vmax=COLOUR_MAXIMUM_CPD,
    )
    axis.set(
        title=title,
        xlabel="Nominal azimuth (degrees)",
        ylabel="Nominal elevation (degrees)",
    )

frequency_boundary_figure.suptitle(
    "Spatial-frequency limits from local resolution"
)
frequency_boundary_figure.colorbar(
    frequency_boundary_image,
    ax=frequency_boundary_axes,
    label="cycles per nominal degree",
)
plt.show()

## 8. Coverage as grating frequency increases

Both curves use the same local resolution. They differ only in whether each cycle is required to contain 2 or 4 pixels.

In [ ]:
# Parameters for this figure
SPATIAL_FREQUENCY_CPD = 0.30
NYQUIST_PIXELS_PER_CYCLE = 2.0
SAFETY_PIXELS_PER_CYCLE = 4.0
CPD_CURVE_RANGE = (0.01, 1.50)
CURVE_POINT_COUNT = 250

coverage_curve_parameters = {
    "selected_spatial_frequency_cpd": SPATIAL_FREQUENCY_CPD,
    "theoretical_pixels_per_cycle": NYQUIST_PIXELS_PER_CYCLE,
    "safety_pixels_per_cycle": SAFETY_PIXELS_PER_CYCLE,
    "cpd_curve_range": list(CPD_CURVE_RANGE),
    "curve_point_count": CURVE_POINT_COUNT,
}

frequency_values_cpd = np.linspace(
    *CPD_CURVE_RANGE,
    CURVE_POINT_COUNT,
)
fraction_satisfying_nyquist_sampling = np.sum(
    (
        (
            cell_pixels_per_degree[np.newaxis, :, :]
            / frequency_values_cpd[:, np.newaxis, np.newaxis]
        )
        > NYQUIST_PIXELS_PER_CYCLE
    )
    * cell_solid_angle_weights[np.newaxis, :, :],
    axis=(1, 2),
) / cell_solid_angle_weights.sum()
fraction_meeting_safety_criterion = np.sum(
    (
        (
            cell_pixels_per_degree[np.newaxis, :, :]
            / frequency_values_cpd[:, np.newaxis, np.newaxis]
        )
        >= SAFETY_PIXELS_PER_CYCLE
    )
    * cell_solid_angle_weights[np.newaxis, :, :],
    axis=(1, 2),
) / cell_solid_angle_weights.sum()

coverage_curve_figure, coverage_curve_axis = plt.subplots(
    figsize=(9, 5),
    constrained_layout=True,
)
coverage_curve_axis.plot(
    frequency_values_cpd,
    100 * fraction_satisfying_nyquist_sampling,
    label=f"{NYQUIST_PIXELS_PER_CYCLE:g} pixels/cycle",
)
coverage_curve_axis.plot(
    frequency_values_cpd,
    100 * fraction_meeting_safety_criterion,
    label=f"{SAFETY_PIXELS_PER_CYCLE:g} pixels/cycle",
)
coverage_curve_axis.axvline(
    SPATIAL_FREQUENCY_CPD,
    color="#555555",
    linestyle="--",
    label=f"Selected frequency: {SPATIAL_FREQUENCY_CPD:g} cpd",
)
coverage_curve_axis.set(
    title="Dome coverage versus grating frequency",
    xlabel="Spatial frequency (cycles per nominal degree)",
    ylabel="Dome surface satisfying check (%)",
    xlim=CPD_CURVE_RANGE,
    ylim=(0, 100),
)
coverage_curve_axis.legend(frameon=False)
plt.gca().spines[['top', 'right']].set_visible(False)

plt.show()

## 9. Save results

Set `SAVE_FIGURES = True` in the settings cell to save the current
figures, tables, and settings in a new timestamped folder.

In [ ]:
# Parameters for saving
SAVE_FIGURES = True
OUTPUT_DIRECTORY = ANALYSIS_DIRECTORY / "outputs"

if SAVE_FIGURES:
    created_utc = datetime.now(UTC)
    run_timestamp = created_utc.strftime("%Y%m%d_%H%M%S_%f")
    run_name = (
        f"{mesh_path.stem}_"
        f"{run_timestamp}"
    )
    run_directory = OUTPUT_DIRECTORY / run_name
    run_directory.mkdir(parents=True, exist_ok=False)

    input_sha256 = hashlib.sha256(mesh_path.read_bytes()).hexdigest()
    analysis_script_path = (
        ANALYSIS_DIRECTORY / "analyse_meshmap_resolution.py"
    )
    notebook_path = (
        ANALYSIS_DIRECTORY / "explore_meshmap_resolution.ipynb"
    )
    saved_settings = {
        "created_utc": created_utc.isoformat(timespec="seconds"),
        "input_file": str(mesh_path),
        "input_sha256": input_sha256,
        "worksheet": selected_sheet_name,
        "analysis_version": ANALYSIS_VERSION,
        "analysis_script_sha256": hashlib.sha256(
            analysis_script_path.read_bytes()
        ).hexdigest(),
        "notebook_sha256": hashlib.sha256(
            notebook_path.read_bytes()
        ).hexdigest(),
        "software_versions": {
            "python": platform.python_version(),
            "numpy": np.__version__,
            "pandas": pd.__version__,
            "matplotlib": version("matplotlib"),
            "plotly": version("plotly"),
            "openpyxl": version("openpyxl"),
        },
        "projector_width_pixels": PROJECTOR_WIDTH_PIXELS,
        "projector_height_pixels": PROJECTOR_HEIGHT_PIXELS,
        "azimuth_range_degrees": list(AZIMUTH_RANGE_DEGREES),
        "elevation_range_degrees": list(ELEVATION_RANGE_DEGREES),
        "view_range_source": (
            "src/main.bonsai and src/VisualResolutionCheck.bonsai "
            "OrthographicView settings"
        ),
        "section_parameters": {
            "summary": summary_parameters,
            "local_resolution": local_resolution_parameters,
            "sampling_checks": sampling_check_parameters,
            "frequency_boundaries": frequency_boundary_parameters,
            "coverage_curve": coverage_curve_parameters,
        },
    }

    projector_footprint_figure.savefig(
        run_directory / "projector_footprint.png",
        dpi=180,
    )
    local_resolution_figure.savefig(
        run_directory / "local_resolution_flat.png",
        dpi=180,
    )
    horizontal_grating_resolution_figure.savefig(
        run_directory / "horizontal_grating_resolution_flat.png",
        dpi=180,
    )
    vertical_grating_resolution_figure.savefig(
        run_directory / "vertical_grating_resolution_flat.png",
        dpi=180,
    )
    sampling_check_figure.savefig(
        run_directory / "sampling_checks_flat.png",
        dpi=180,
    )
    frequency_boundary_figure.savefig(
        run_directory / "frequency_boundaries_flat.png",
        dpi=180,
    )
    coverage_curve_figure.savefig(
        run_directory / "coverage_curve.png",
        dpi=180,
    )
    local_resolution_dome.write_html(
        run_directory / "local_resolution_dome.html"
    )
    horizontal_grating_resolution_dome.write_html(
        run_directory / "horizontal_grating_resolution_dome.html"
    )
    vertical_grating_resolution_dome.write_html(
        run_directory / "vertical_grating_resolution_dome.html"
    )
    theoretical_limit_dome.write_html(
        run_directory / "theoretical_limit_dome.html"
    )
    safety_criterion_dome.write_html(
        run_directory / "safety_criterion_dome.html"
    )
    nyquist_frequency_boundary_dome.write_html(
        run_directory / "nyquist_frequency_boundary_dome.html"
    )
    safety_frequency_boundary_dome.write_html(
        run_directory / "safety_frequency_boundary_dome.html"
    )
    resolution_summary.to_frame().to_csv(
        run_directory / "resolution_summary.csv"
    )
    sampling_summary.to_csv(
        run_directory / "sampling_summary.csv",
        index=False,
    )
    (run_directory / "settings.json").write_text(
        json.dumps(saved_settings, indent=2),
        encoding="utf-8",
    )
    print(f"Saved results to: {run_directory}")
else:
    print(
        "Nothing saved. Set SAVE_FIGURES = True to save the results."
    )